# Import Libraries

In [16]:
import ast
import pandas as pd
from collections import Counter
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load 3 CSV files

In [18]:
train_raw = pd.read_csv(
    r"D:\CSE DEPT DOCUMENT\11th Semester\THESIS\CLEANED DATASET\Training_clean.csv",
    usecols=["id", "acts", "emotions", "utterances", "context_state"]
)
test_raw = pd.read_csv(
    r"D:\CSE DEPT DOCUMENT\11th Semester\THESIS\CLEANED DATASET\Testing_clean.csv",
    usecols=["id", "acts", "emotions", "utterances", "context_state"]
)
val_raw = pd.read_csv(
    r"D:\CSE DEPT DOCUMENT\11th Semester\THESIS\CLEANED DATASET\Validation_clean.csv",
    usecols=["id", "acts", "emotions", "utterances", "context_state"]
)

print("Loaded → Train:", len(train_raw))
print("Loaded → Test :", len(test_raw))
print("Loaded → Val  :", len(val_raw))

Loaded → Train: 11118
Loaded → Test : 1000
Loaded → Val  : 1000


# Helper

In [19]:
def majority_label(x):
    """Parse string→list if needed, then return majority label as string."""
    if isinstance(x, str):
        x = ast.literal_eval(x)
    if isinstance(x, (list, tuple)) and len(x) > 0:
        return str(Counter(x).most_common(1)[0][0])
    return str(x)

# Feature & Target Creation

In [20]:
def prepare_data(df):
    df = df.copy()

    # FULL conversation feature
    df["conversation_text"] = df["utterances"].apply(
        lambda x: " ".join(ast.literal_eval(x) if isinstance(x, str) else x)
    )

    # Emotion Target = majority emotion label (integer→string)
    df["emotion_target"] = df["emotions"].apply(majority_label)

    # Act Target = majority act label (integer→string)
    df["act_target"] = df["acts"].apply(majority_label)

    # Context Target = already scalar string
    df["context_target"] = df["context_state"].astype(str)

    return df


train = prepare_data(train_raw)
valid = prepare_data(val_raw)
test  = prepare_data(test_raw)

# Dataset Split Sizes (Train / Validation / Test)

In [21]:
print(train.shape)
print(valid.shape)
print(test.shape)

(11118, 9)
(1000, 9)
(1000, 9)


# Dataset Column Names

In [22]:
print(train.columns.tolist())

['id', 'acts', 'emotions', 'utterances', 'context_state', 'conversation_text', 'emotion_target', 'act_target', 'context_target']


# Verify

In [23]:
print("\n=== Verify Targets ===")
print("emotion_target :", repr(train["emotion_target"].iloc[0]), type(train["emotion_target"].iloc[0]))
print("act_target     :", repr(train["act_target"].iloc[0]),     type(train["act_target"].iloc[0]))
print("context_target :", repr(train["context_target"].iloc[0]), type(train["context_target"].iloc[0]))

assert train["emotion_target"].apply(lambda x: isinstance(x, str)).all()
assert train["act_target"].apply(lambda x: isinstance(x, str)).all()
assert train["context_target"].apply(lambda x: isinstance(x, str)).all()
print("\n✅ সব target ঠিক আছে — SVM/RF চালানো যাবে।")


=== Verify Targets ===
emotion_target : '0' <class 'str'>
act_target     : '3' <class 'str'>
context_target : 'achievement' <class 'str'>

✅ সব target ঠিক আছে — SVM/RF চালানো যাবে।


# Tasks

In [25]:
tasks = {
    "Emotion": "emotion_target",
    "Act":     "act_target",
    "Context": "context_target"
}

# Dictionary to store all results for final summary

In [26]:
results_summary = {}

# Quick sanity-check: confirm all targets are now scalar strings

In [27]:
print("\nSample emotion_target:", train["emotion_target"].iloc[0])
print("Sample act_target    :", train["act_target"].iloc[0])
print("Sample context_target:", train["context_target"].iloc[0])


Sample emotion_target: 0
Sample act_target    : 3
Sample context_target: achievement


# Model Train

In [29]:
for task_name, target_col in tasks.items():

    print("\n" + "="*70)
    print(f"TASK: {task_name}")
    print("="*70)

    X_train = train["conversation_text"]
    y_train = train[target_col]

    X_valid = valid["conversation_text"]
    y_valid = valid[target_col]

    X_test = test["conversation_text"]
    y_test = test[target_col]

    # ================= SVM =================
    svm_model = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf",   LinearSVC(random_state=42))
    ])

    svm_model.fit(X_train, y_train)

    train_pred_svm = svm_model.predict(X_train)
    valid_pred_svm = svm_model.predict(X_valid)
    test_pred_svm  = svm_model.predict(X_test)

    svm_train_acc = accuracy_score(y_train, train_pred_svm)
    svm_valid_acc = accuracy_score(y_valid, valid_pred_svm)
    svm_test_acc  = accuracy_score(y_test,  test_pred_svm)

    print("\n----- SVM -----")
    print("Training Accuracy  :", svm_train_acc)
    print("Validation Accuracy:", svm_valid_acc)
    print("Test Accuracy      :", svm_test_acc)
    print("\nClassification Report (Test)")
    print(classification_report(y_test, test_pred_svm))

    # ================= RF =================
    rf_model = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000)),
        ("clf",   RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ))
    ])

    rf_model.fit(X_train, y_train)

    train_pred_rf = rf_model.predict(X_train)
    valid_pred_rf = rf_model.predict(X_valid)
    test_pred_rf  = rf_model.predict(X_test)

    rf_train_acc = accuracy_score(y_train, train_pred_rf)
    rf_valid_acc = accuracy_score(y_valid, valid_pred_rf)
    rf_test_acc  = accuracy_score(y_test,  test_pred_rf)

    print("\n----- Random Forest -----")
    print("Training Accuracy  :", rf_train_acc)
    print("Validation Accuracy:", rf_valid_acc)
    print("Test Accuracy      :", rf_test_acc)
    print("\nClassification Report (Test)")
    print(classification_report(y_test, test_pred_rf))

    results_summary[task_name] = {
        "SVM": {
            "Train":      svm_train_acc,
            "Validation": svm_valid_acc,
            "Test":       svm_test_acc
        },
        "RandomForest": {
            "Train":      rf_train_acc,
            "Validation": rf_valid_acc,
            "Test":       rf_test_acc
        }
    }



TASK: Emotion

----- SVM -----
Training Accuracy  : 0.9707681237632668
Validation Accuracy: 0.957
Test Accuracy      : 0.898

Classification Report (Test)
              precision    recall  f1-score   support

           0       0.91      0.98      0.95       890
           1       0.50      0.09      0.15        11
           2       0.00      0.00      0.00         5
           3       0.00      0.00      0.00         2
           4       0.59      0.32      0.42        84
           5       0.00      0.00      0.00         8
           6       0.00      0.00      0.00         0

    accuracy                           0.90      1000
   macro avg       0.29      0.20      0.22      1000
weighted avg       0.87      0.90      0.88      1000



d:\CSE DEPT DOCUMENT\11th Semester\THESIS\CODE\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\CSE DEPT DOCUMENT\11th Semester\THESIS\CODE\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\CSE DEPT DOCUMENT\11th Semester\THESIS\CODE\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier


----- Random Forest -----
Training Accuracy  : 0.9919949631228638
Validation Accuracy: 0.958
Test Accuracy      : 0.891

Classification Report (Test)
              precision    recall  f1-score   support

           0       0.90      0.99      0.94       890
           1       0.00      0.00      0.00        11
           2       0.00      0.00      0.00         5
           3       0.00      0.00      0.00         2
           4       0.61      0.17      0.26        84
           5       0.00      0.00      0.00         8
           6       0.00      0.00      0.00         0

    accuracy                           0.89      1000
   macro avg       0.22      0.16      0.17      1000
weighted avg       0.85      0.89      0.86      1000


TASK: Act


d:\CSE DEPT DOCUMENT\11th Semester\THESIS\CODE\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\CSE DEPT DOCUMENT\11th Semester\THESIS\CODE\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\CSE DEPT DOCUMENT\11th Semester\THESIS\CODE\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier


----- SVM -----
Training Accuracy  : 0.9149127540924626
Validation Accuracy: 0.591
Test Accuracy      : 0.668

Classification Report (Test)
              precision    recall  f1-score   support

           1       0.72      0.79      0.76       533
           2       0.59      0.54      0.56       272
           3       0.60      0.51      0.55       195

    accuracy                           0.67      1000
   macro avg       0.64      0.61      0.62      1000
weighted avg       0.66      0.67      0.66      1000


----- Random Forest -----
Training Accuracy  : 0.9882173052707321
Validation Accuracy: 0.561
Test Accuracy      : 0.657

Classification Report (Test)
              precision    recall  f1-score   support

           1       0.64      0.95      0.77       533
           2       0.71      0.36      0.48       272
           3       0.70      0.28      0.40       195

    accuracy                           0.66      1000
   macro avg       0.68      0.53      0.55      1000
w

d:\CSE DEPT DOCUMENT\11th Semester\THESIS\CODE\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\CSE DEPT DOCUMENT\11th Semester\THESIS\CODE\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\CSE DEPT DOCUMENT\11th Semester\THESIS\CODE\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 


----- Random Forest -----
Training Accuracy  : 0.9989206691851052
Validation Accuracy: 0.767
Test Accuracy      : 0.738

Classification Report (Test)
                       precision    recall  f1-score   support

    academic_pressure       0.00      0.00      0.00         8
          achievement       0.75      0.73      0.74       427
    emotional_support       0.80      0.36      0.50        11
     financial_stress       0.00      0.00      0.00         6
       health_concern       1.00      0.47      0.64        15
           loneliness       1.00      0.67      0.80         3
      ongoing_problem       0.71      0.85      0.77       456
relationship_conflict       1.00      0.34      0.51        74

             accuracy                           0.74      1000
            macro avg       0.66      0.43      0.49      1000
         weighted avg       0.75      0.74      0.72      1000



d:\CSE DEPT DOCUMENT\11th Semester\THESIS\CODE\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\CSE DEPT DOCUMENT\11th Semester\THESIS\CODE\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\CSE DEPT DOCUMENT\11th Semester\THESIS\CODE\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 

# FINAL SUMMARY

In [30]:
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

for task_name, models in results_summary.items():
    print(f"\nTask: {task_name}")
    for model_name, scores in models.items():
        print(f"  {model_name}:")
        print(f"    Training Accuracy   : {scores['Train']*100:.2f}%")
        print(f"    Validation Accuracy : {scores['Validation']*100:.2f}%")
        print(f"    Test Accuracy       : {scores['Test']*100:.2f}%")


FINAL SUMMARY

Task: Emotion
  SVM:
    Training Accuracy   : 97.08%
    Validation Accuracy : 95.70%
    Test Accuracy       : 89.80%
  RandomForest:
    Training Accuracy   : 99.20%
    Validation Accuracy : 95.80%
    Test Accuracy       : 89.10%

Task: Act
  SVM:
    Training Accuracy   : 91.49%
    Validation Accuracy : 59.10%
    Test Accuracy       : 66.80%
  RandomForest:
    Training Accuracy   : 98.82%
    Validation Accuracy : 56.10%
    Test Accuracy       : 65.70%

Task: Context
  SVM:
    Training Accuracy   : 95.00%
    Validation Accuracy : 80.00%
    Test Accuracy       : 79.00%
  RandomForest:
    Training Accuracy   : 99.89%
    Validation Accuracy : 76.70%
    Test Accuracy       : 73.80%


# DailyDialog Dataset Sample Inspection (Utterances, Acts, Emotions, Context)

In [31]:
for i in range(5):

    print("\n" + "="*60)
    print("SAMPLE:", i)
    print("="*60)

    print("\nRAW utterances:")
    print(train["utterances"].iloc[i])

    print("\nPROCESSED conversation_text:")
    print(train["conversation_text"].iloc[i])

    print("\nRAW emotions:")
    print(train["emotions"].iloc[i])

    print("\nEmotion Target:")
    print(train["emotion_target"].iloc[i])

    print("\nRAW acts:")
    print(train["acts"].iloc[i])

    print("\nAct Target:")
    print(train["act_target"].iloc[i])

    print("\nContext State:")
    print(train["context_state"].iloc[i])

    print("\nContext Target:")
    print(train["context_target"].iloc[i])


SAMPLE: 0

RAW utterances:
["Say , Jim , how about going for a few beers after dinner ?", "You know that is tempting but is really not good for our fitness .", "What do you mean ? It will help us to relax .", "Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ?", "I guess you are right.But what shall we do ? I don't feel like sitting at home .", "I suggest a walk over to the gym where we can play singsong and meet some of our friends .", "That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them .", "Sounds great to me ! If they are willing , we could ask them to go dancing with us.That is excellent exercise and fun , too .", "Good.Let ' s go now .", "All right ."]

PROCESSED conversation_text:
Say , Jim , how about going for a few beers after dinner ? You know that is tempting but is really not good for our fitness . What do you mean ? It will help us to relax . Do you really think 